# Cell composition and UMAP starting-position scores
Relate each session's neuron-type composition to its **saved starting-position regression R? on UMAP1?UMAP3**. Analyze tracking/playback and raw/baseline-subtracted activity separately. Each dot is a session entry; do not treat bouts/windows as independent samples here.

This is an observational, descriptive association analysis. It does not show that changing composition causes a change in performance. Scores are the earlier in-sample regressions, not held-out accuracy. Animal/headstage, total unit count, behavioral sampling, missing classifications and UMAP seed can all affect the comparison. No UMAP or starting-position regression is refitted.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib import reload
import hashlib
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display
cwd = Path.cwd().resolve()
repo_dir = next((candidate for parent in (cwd, *cwd.parents) for candidate in (parent, parent / "etheremin_project")
                 if (candidate / "movement_bouts" / "evaluation.py").is_file()), None)
if repo_dir is None:
    raise FileNotFoundError("Run from the eTheremin workspace or inside the repository.")
sys.path.insert(0, str(repo_dir / "movement_bouts"))
import evaluation
reload(evaluation)
DATA_DIR = repo_dir.parent / "data"
COMPOSITION_PATH = DATA_DIR / "derived/neuron_type_composition/07f355d4bb69/composition.pkl"
UMAP_RUN_DIR = DATA_DIR / "derived/premotor_activity/runs/umap_overall_k30_seed42_fdb86ddd0e53"
SCORE = "r2_adjusted"  # "r2" for ordinary R?; matches earlier adjusted-R? plots by default.
MIN_CLASSIFICATION_COVERAGE = 0.0  # Fraction in [0,1]; always require at least one classified unit.
SAVE_RESULTS = True
TYPES = ["Motor", "Sensory", "Sensorimotor", "None"]
FRACTIONS = {name: f"fraction_{name}" for name in TYPES}
COLORS = evaluation.NEURON_TYPE_COLORS
NUMERIC_CONTROLS = ["n_total", "classification_coverage", "log_n_bouts", "position_std", "abs_side_bias"]
CATEGORICAL_CONTROLS = ["animal_headstage"]


## Load and match the saved results
Choose one explicit UMAP run above; never mix seeds, binning, preprocessing or regression filters. Select evaluation files whose metadata identifies UMAP and starting position. If multiple different evaluations exist for one activity mode, choose the intended files explicitly rather than silently taking the latest.

Validate original session IDs, animal/headstage identity, extraction provenance and exact fitted neuron membership. Composition is counted once per session, then joined to the two condition-specific scores.


In [ ]:
if SCORE not in ("r2", "r2_adjusted") or not 0 <= MIN_CLASSIFICATION_COVERAGE <= 1:
    raise ValueError("Choose r2/r2_adjusted and coverage between zero and one.")
composition = pd.read_pickle(COMPOSITION_PATH)
counts = composition["session_counts"].copy().sort_index().rename_axis("idx")
for filename, expected_hash in composition["settings"]["source_sha256"].items():
    if hashlib.sha256(Path(filename).read_bytes()).hexdigest() != expected_hash:
        raise ValueError(f"Composition source changed: {filename}; regenerate the count notebook.")
counts["classification_coverage"] = counts["n_classified"] / counts["n_total"].replace(0, np.nan)
for category, column in FRACTIONS.items():
    counts[column] = counts[category] / counts["n_classified"].replace(0, np.nan)
counts["animal_headstage"] = counts["animal"].astype(str) + " / hs" + counts["headstage"].astype(str)
counts["idx"] = counts.index
score_bundles, score_paths = {}, {}
for path in sorted((UMAP_RUN_DIR / "evaluation").glob("initial_position_*_pooled.pkl")):
    bundle = pd.read_pickle(path)
    saved = bundle["settings"]
    if saved.get("representation") != "UMAP":
        continue  # Ignore obsolete files whose provenance still says PCA.
    source = saved["representation_settings"]
    if saved["target"] != "initial_position" or saved["min_displacement"] is not None or saved["window_mode"] != "pooled":
        raise ValueError(f"Unexpected target/filter/window mode: {path}")
    if saved["columns"] != ["UMAP1", "UMAP2", "UMAP3"] or source.get("neuron_population") != "all":
        raise ValueError("Use the all-neuron, three-coordinate starting-position evaluation.")
    mode = source["activity_mode"]
    if mode in score_bundles:
        raise ValueError(f"Multiple UMAP evaluations for {mode}; select one explicit file.")
    score_bundles[mode], score_paths[mode] = bundle, path
if set(score_bundles) != {"raw", "baseline_subtracted"}:
    raise ValueError("Need raw and baseline-subtracted UMAP starting-position evaluations.")
for mode, bundle in score_bundles.items():
    source = bundle["settings"]["representation_settings"]
    if source["extraction_parameters"]["source_sha256"]["session_info.pkl"] != composition["settings"]["source_sha256"][str(DATA_DIR / "derived/movement_bouts/session_info.pkl")]:
        raise ValueError("UMAP and composition use different source session metadata.")
    if not bundle["summary"].index.is_unique:
        raise ValueError("Duplicate session/condition scores.")
    for field in ("n_neighbors", "min_dist", "random_state", "N_BINS", "analysis_window_s", "baseline_window_s"):
        if source[field] != score_bundles["raw"]["settings"]["representation_settings"][field]:
            raise ValueError(f"UMAP modes differ in {field}.")
    for field in ("animal", "headstage", "session"):
        observed = bundle["summary"].reset_index().set_index("idx")[field]
        if not observed.eq(counts[field].reindex(observed.index)).all():
            raise ValueError(f"Score and composition {field} identity mismatch.")
# Audit exact membership against each saved embedding, not just the total count.
score_indices = sorted(set().union(*(set(b["summary"].index.get_level_values("idx")) for b in score_bundles.values())))
for idx in score_indices:
    expected = composition["neuron_inventory"].loc[lambda t: t["idx"].eq(idx)].sort_values("neuron_axis")["neuron_id"].tolist()
    session_bundle = pd.read_pickle(UMAP_RUN_DIR / "sessions" / f"session_{idx}.pkl")
    for mode, bundle in score_bundles.items():
        for condition in (0, 1):
            if (idx, condition) in bundle["fits"]:
                fitted = session_bundle["fits"][mode][condition]
                if fitted["neuron_ids"] != expected:
                    raise ValueError(f"Fitted neuron membership mismatch: {idx}, {mode}, {condition}.")
                if not bundle["fits"][(idx, condition)]["predictions"].index.isin(fitted["results"].index).all():
                    raise ValueError("Regression observations do not match saved UMAP observations.")
frames = []
for mode, bundle in score_bundles.items():
    # Behavioral spread/bias use actual fitted bouts, counted once per bout.
    summary = evaluation.summarize_starting_position_bias(bundle)
    summary = summary.drop(columns=[name for name in ("animal", "headstage", "session", "region") if name in summary])
    joined = summary.reset_index().merge(counts.drop(columns="idx").reset_index(), on="idx", how="left", validate="many_to_one")
    joined["activity_mode"] = mode
    joined["log_n_bouts"] = np.log1p(joined["n_bouts"])
    joined["eligible_score"] = joined["status"].eq("ok") & np.isfinite(joined[SCORE])
    joined["eligible_composition"] = joined["n_classified"].gt(0) & joined["classification_coverage"].ge(MIN_CLASSIFICATION_COVERAGE)
    frames.append(joined)
session_scores = pd.concat(frames, ignore_index=True).set_index(["activity_mode", "idx", "condition"]).sort_index()
eligibility = session_scores[["animal", "headstage", "n_total", "n_classified", "classification_coverage", "eligible_score", "eligible_composition"]].copy()
display(eligibility.groupby(level=["activity_mode", "condition"])[["eligible_score", "eligible_composition"]].sum())
display(eligibility.loc[~eligibility["eligible_composition"]].drop_duplicates())
print("UMAP settings:", {k: source[k] for k in ("n_neighbors", "min_dist", "random_state", "N_BINS", "analysis_window_s")})


## Composition proportions and starting-position R?
Fractions below are **among classified units**, with all-unclassified sessions excluded. Each panel uses one score per session; marker shape identifies animal/headstage. Cell-type colors match the other notebooks. Spearman correlations are descriptive summaries without significance tests.

Type fractions sum to one. Each type is examined separately; a higher fraction necessarily implies a lower combined fraction of other types. Missing coverage remains a separate measurement, not the None class.


In [ ]:
panels = [(mode, condition) for mode in ("raw", "baseline_subtracted") for condition in (0, 1)]
panel_names = [f"{mode.replace('_', ' ')} | {'Tracking' if condition == 0 else 'Playback'}" for mode, condition in panels]
blocks = counts["animal_headstage"].drop_duplicates().tolist()
markers = dict(zip(blocks, ["o", "s", "^", "D", "P", "X", "v", "<"]))
if len(markers) < len(blocks):
    raise ValueError("Add marker shapes for the extra animal/headstage blocks.")
fig_composition, axes = plt.subplots(4, 4, figsize=(17, 14), sharey=True, layout="constrained")
for col, ((mode, condition), title) in enumerate(zip(panels, panel_names)):
    rows = session_scores.xs((mode, condition), level=("activity_mode", "condition"))
    rows = rows.loc[rows["eligible_score"] & rows["eligible_composition"]]
    for row, (category, fraction) in enumerate(FRACTIONS.items()):
        ax = axes[row, col]
        for block in blocks:
            subset = rows.loc[rows["animal_headstage"].eq(block)]
            ax.scatter(100 * subset[fraction], subset[SCORE], color=COLORS[category], marker=markers[block],
                       s=32, alpha=.8, edgecolor=".25", linewidth=.4)
        rho = rows[fraction].rank().corr(rows[SCORE].rank())
        ax.text(.03, .96, f"n={len(rows)}; Spearman r={rho:.2f}", transform=ax.transAxes, va="top", fontsize=8)
        ax.axhline(0, color=".7", linestyle=":", linewidth=.8)
        ax.set(xlabel=f"{category} (% of classified units)", ylabel=("Adjusted R-squared" if SCORE == "r2_adjusted" else "R-squared") if col == 0 else "", title=title if row == 0 else "")
fig_composition.legend(handles=[Line2D([], [], color=".35", marker=markers[b], linestyle="", label=b) for b in blocks],
                       loc="outside lower center", ncol=3, frameon=False)
fig_composition.suptitle("Recorded cell composition versus UMAP starting-position score")
plt.show()


## Check total neuron count and classification coverage
These panels include all valid session scores, including entirely unclassified sessions. Classification coverage is missing metadata about types; it is not itself a biological type. Total units and animal/headstage may help explain apparent pooled composition relationships.


In [ ]:
fig_diagnostics, axes = plt.subplots(2, 4, figsize=(16, 8), sharey=True, layout="constrained")
for col, ((mode, condition), title) in enumerate(zip(panels, panel_names)):
    rows = session_scores.xs((mode, condition), level=("activity_mode", "condition"))
    rows = rows.loc[rows["eligible_score"]]
    for row, (column, label, scale) in enumerate((("n_total", "Total recorded neurons", 1), ("classification_coverage", "Classification coverage (%)", 100))):
        ax = axes[row, col]
        for block in blocks:
            subset = rows.loc[rows["animal_headstage"].eq(block)]
            ax.scatter(scale * subset[column], subset[SCORE], color="red" if condition == 0 else "black",
                       marker=markers[block], s=30, alpha=.65)
        ax.set(xlabel=label, ylabel=("Adjusted R-squared" if SCORE == "r2_adjusted" else "R-squared") if col == 0 else "", title=title if row == 0 else "")
        ax.axhline(0, color=".7", linestyle=":", linewidth=.8)
fig_diagnostics.legend(handles=[Line2D([], [], color=".35", marker=markers[b], linestyle="", label=b) for b in blocks],
                       loc="outside lower center", ncol=3, frameon=False)
fig_diagnostics.suptitle("Potential confounds: total neurons and classification coverage")
plt.show()


## Association after accounting for recording block and sampling
For **each type separately**, fit a descriptive session-level linear model:

`starting-position score ~ type fraction + total neuron count + classification coverage + log(1 + bout count) + starting-position spread + absolute starting-side bias + animal/headstage indicators`.

The controls-only and full models use identical sessions. Coefficients report a **10 percentage-point increase** in the classified fraction. `partial_r` correlates the score and fraction after both are residualized against the controls. `added_session_r2` is the gain in the *across-session association model's* R?, distinct from the underlying UMAP starting-position R?. These are in-sample descriptive fits, with no p-values, confidence intervals or causal interpretation. Residual degrees of freedom do not make sessions from the same animal independent.

Repeat on fully classified sessions as a coverage sensitivity check. A coefficient can change when recording-block or behavioral sampling differences are accounted for; collinear or saturated designs are explicitly skipped.


In [ ]:
associations, complete_coverage_associations = {}, {}
for mode, condition in panels:
    rows = session_scores.xs((mode, condition), level=("activity_mode", "condition"))
    selected = rows.loc[rows["eligible_score"] & rows["eligible_composition"]]
    associations[(mode, condition)] = evaluation.fit_session_composition_associations(
        selected, score=SCORE, fractions=FRACTIONS, numeric_controls=NUMERIC_CONTROLS,
        categorical_controls=CATEGORICAL_CONTROLS,
    )
    complete_coverage_associations[(mode, condition)] = evaluation.fit_session_composition_associations(
        selected.loc[selected["classification_coverage"].eq(1)], score=SCORE, fractions=FRACTIONS,
        numeric_controls=NUMERIC_CONTROLS, categorical_controls=CATEGORICAL_CONTROLS,
    )
association_summary = pd.concat({key: value["summary"] for key, value in associations.items()}, names=["activity_mode", "condition"])
complete_coverage_summary = pd.concat({key: value["summary"] for key, value in complete_coverage_associations.items()}, names=["activity_mode", "condition"])
display(association_summary)
print("Fully classified sessions only:")
display(complete_coverage_summary[["n_sessions", "coefficient_per_10pp", "partial_r", "added_session_r2", "status", "reason"]])
fig_adjusted, axes = plt.subplots(1, 4, figsize=(16, 4.5), sharey=True, layout="constrained")
for ax, key, title in zip(axes, panels, panel_names):
    summary = associations[key]["summary"]
    for i, category in enumerate(TYPES):
        row = summary.loc[category]
        if row["status"] == "ok":
            ax.scatter(i, row["coefficient_per_10pp"], color=COLORS[category], edgecolor=".25", s=80)
    ax.axhline(0, color=".5", linestyle="--", linewidth=.8)
    ax.set(title=title, xticks=range(len(TYPES)), xticklabels=TYPES, ylabel="Score change per +10 percentage points" if ax is axes[0] else "")
    ax.tick_params(axis="x", rotation=35)
fig_adjusted.suptitle("Composition associations after accounting for recording block and sampling")
plt.show()


## Does composition relate to the tracking advantage?
Match tracking and playback within a session and use **tracking minus playback score** as the outcome. A session has the same neuron composition in both conditions. Analyze the two activity modes separately, with block, neuron count and coverage controls plus differences in bout count, position spread and side bias. This asks whether composition is associated with the *size of the condition difference*, rather than counting the same session twice.


In [ ]:
paired_tables, paired_associations = {}, {}
paired_controls = ["n_total", "classification_coverage", "delta_log_n_bouts", "delta_position_std", "delta_abs_side_bias"]
for mode in ("raw", "baseline_subtracted"):
    rows = session_scores.xs(mode, level="activity_mode")
    rows = rows.loc[rows["eligible_score"] & rows["eligible_composition"]]
    tracking = rows.xs(0, level="condition")
    playback = rows.xs(1, level="condition")
    common = tracking.index.intersection(playback.index)
    paired = tracking.loc[common].copy()
    paired["score_difference"] = tracking.loc[common, SCORE] - playback.loc[common, SCORE]
    for column in ("log_n_bouts", "position_std", "abs_side_bias"):
        paired[f"delta_{column}"] = tracking.loc[common, column] - playback.loc[common, column]
    for column in [*FRACTIONS.values(), "n_total", "classification_coverage"]:
        if not tracking.loc[common, column].equals(playback.loc[common, column]):
            raise ValueError(f"Within-session composition mismatch: {column}")
    paired_tables[mode] = paired
    paired_associations[mode] = evaluation.fit_session_composition_associations(
        paired, score="score_difference", fractions=FRACTIONS,
        numeric_controls=paired_controls, categorical_controls=CATEGORICAL_CONTROLS,
    )
paired_summary = pd.concat({mode: result["summary"] for mode, result in paired_associations.items()}, names=["activity_mode"])
display(paired_summary)
fig_paired, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True, layout="constrained")
for ax, (mode, result) in zip(axes, paired_associations.items()):
    for i, category in enumerate(TYPES):
        row = result["summary"].loc[category]
        if row["status"] == "ok":
            ax.scatter(i, row["coefficient_per_10pp"], color=COLORS[category], edgecolor=".25", s=80)
    ax.axhline(0, color=".5", linestyle="--", linewidth=.8)
    ax.set(title=mode.replace("_", " "), xticks=range(len(TYPES)), xticklabels=TYPES,
           ylabel=("Change in tracking - playback score" + chr(10) + "per +10 percentage points") if ax is axes[0] else "")
    ax.tick_params(axis="x", rotation=25)
fig_paired.suptitle("Composition and tracking advantage after controls")
plt.show()


## Save the matched data, models and figures
Preserve the source run, score selection, classification provenance, joined session table, model inputs/residuals, sensitivity checks and skip reasons. Original performance and composition files remain unchanged. A useful next validation would be repeating this comparison across UMAP seeds; this notebook describes only the saved realization.


In [ ]:
analysis_settings = {
    "score": SCORE, "fractions": FRACTIONS, "fraction_denominator": "classified neurons only",
    "min_classification_coverage": MIN_CLASSIFICATION_COVERAGE,
    "numeric_controls": NUMERIC_CONTROLS, "categorical_controls": CATEGORICAL_CONTROLS,
    "composition_path": str(COMPOSITION_PATH), "umap_run_dir": str(UMAP_RUN_DIR),
    "source_sha256": {str(path): hashlib.sha256(path.read_bytes()).hexdigest()
                      for path in [COMPOSITION_PATH, *score_paths.values()]},
    "score_settings": {mode: bundle["settings"] for mode, bundle in score_bundles.items()},
    "scoring": "descriptive_in_sample_associations", "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
analysis_bundle = {"settings": analysis_settings, "session_scores": session_scores, "eligibility": eligibility,
                   "associations": associations, "association_summary": association_summary,
                   "complete_coverage_associations": complete_coverage_associations,
                   "complete_coverage_summary": complete_coverage_summary, "paired_tables": paired_tables,
                   "paired_associations": paired_associations, "paired_summary": paired_summary}
if SAVE_RESULTS:
    effective = {k: v for k, v in analysis_settings.items() if k != "created_at_utc"}
    digest = hashlib.sha256(json.dumps(effective, sort_keys=True).encode()).hexdigest()[:12]
    output_dir = DATA_DIR / "derived" / "umap_composition" / digest
    output_dir.mkdir(parents=True, exist_ok=True)
    pd.to_pickle(analysis_bundle, output_dir / "analysis.pkl")
    for name in ("session_scores", "eligibility", "association_summary", "complete_coverage_summary", "paired_summary"):
        analysis_bundle[name].to_pickle(output_dir / f"{name}.pkl")
    (output_dir / "parameters.json").write_text(json.dumps(analysis_settings, indent=2), encoding="utf-8")
    for name, fig in (("composition_scatter", fig_composition), ("count_coverage_checks", fig_diagnostics),
                      ("controlled_associations", fig_adjusted), ("tracking_advantage", fig_paired)):
        fig.savefig(output_dir / f"{name}.png", dpi=180, bbox_inches="tight")
    print(f"Saved analysis and figures: {output_dir}")
